In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

from plotly.subplots import make_subplots
import plotly.graph_objects as go


In [ ]:
# Parameters
gamma = 1/7
beta = 5 * gamma
theta = 1 / 365
mu = 1/(50*365)
m = mu

# Time interval
t_span = (0,50000)
t_eval = np.linspace(t_span[0], t_span[1], 500000)

In [ ]:
beta / gamma

In [ ]:
# ODE system
def model(t, y):
    I, R = y
    dI_dt = I * (beta * (1 - I - R) - gamma - mu)
    dR_dt = gamma * I - theta*R - mu* R
    return [dI_dt, dR_dt]

In [ ]:
# Initial conditions
I0 = 0.01  # small initial infection
R0 = 0.0   # no recovered initially
initial_conditions = [I0, R0]

# Solve the system
sol = solve_ivp(model, t_span, initial_conditions, t_eval=t_eval)

In [ ]:
# Extract solutions
t = sol.t
I = sol.y[0]
R = sol.y[1]

In [ ]:
mu = 0
t_span = (0, 2000)
t_eval = np.linspace(t_span[0], t_span[1], 20000)
sol_mu_zero =  solve_ivp(model, t_span, initial_conditions, t_eval=t_eval)

In [ ]:
# Extract solutions
t_muzero = sol_mu_zero.t
I_mu_zero = sol_mu_zero.y[0]
R_mu_zero = sol_mu_zero.y[1]

In [ ]:
# Troncamento delle serie temporali
x_full = t
y_full = I
y_ref_full = [I_mu_zero[-1]] * len(y_full)

x_post = t[3000:]
y_post = I[3000:]
y_ref_post = [I_mu_zero[-1]] * len(y_post)

# Creazione della figura con subplot (2 colonne)
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.4, 0.6],  # subplot sinistro più stretto
)

# -------------------
# Subplot 1: range iniziale [0:2000]
fig.add_trace(go.Scatter(
    x=x_full[:3000], y=y_full[:3000],
    mode='lines',
    name='I(t) - early',
    line=dict(color='rgb(0, 0, 200)')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x_full[:3000], y=y_ref_full[:3000],
    mode='lines',
    name='Ī - early',
    line=dict(dash='dash', color='red')
), row=1, col=1)

# -------------------
# Subplot 2: range [2000:]
fig.add_trace(go.Scatter(
    x=x_post, y=y_post,
    mode='lines',
    name='I(t) - late',
    line=dict(color='rgb(0, 0, 200)')
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_post, y=y_ref_post,
    mode='lines',
    name='Ī - late',
    line=dict(dash='dash', color='red')
), row=1, col=2)

# -------------------
# Layout
fig.update_layout(
    width=1000,
    height=600,
    title_text=f'μ: {round(m, 8)}    -     θ: {round(theta, 8)}     -     γ: {round(gamma, 3)}',
    title_x=0.5,
    template='simple_white',
    showlegend=True,  # <--- ABILITATO
)

# Annotazioni per sostituire i titoli dei subplot (più in basso)
fig.add_annotation(
    text="Up to t=3000",
    xref='paper', yref='paper',
    x=0.2, y=1.01,  # y abbassata
    showarrow=False,
    font=dict(size=14)
)

fig.add_annotation(
    text="From t=3000 onwards",
    xref='paper', yref='paper',
    x=0.8, y=1.01,
    showarrow=False,
    font=dict(size=14)
)

# Annotazioni
fig.add_annotation(
    text=f'Ī: {round(I_mu_zero[-1], 8)}   -    Iₑ: {round(I[-1], 8)}    -    Iₑ / Ī: {round((beta - mu - gamma)/(beta - gamma)*(1+mu*gamma/(theta**2 + theta*(mu + gamma))), 5)}',
    xref='paper', yref='paper',
    x=0.5, y=1.10,
    showarrow=False,
    font=dict(size=13)
)

# Etichette degli assi
fig.update_xaxes(title_text="Time (days)", row=1, col=1)
fig.update_yaxes(title_text="Infected (I)", row=1, col=1)
fig.update_xaxes(title_text="Time (days)", row=1, col=2)
fig.update_yaxes(title_text="Infected (I)", row=1, col=2)

# Mostra il grafico
fig.show()


In [ ]:
fig = go.Figure()

# Traccia della curva I(t)
fig.add_trace(go.Scatter(
    x=I[1500:-1], y=R[1500:-1],
    mode='lines',
    name='I(t)',
))
fig.add_trace(go.Scatter(
    x=I_mu_zero[1500:], y=R_mu_zero[1500:],
    mode='lines',
    name='I with μ=0 ',
))
# Traccia della linea di riferimento costante
fig.add_trace(go.Scatter(
    x=[I_mu_zero[-1]], y=[R_mu_zero[-1]],
    mode='markers',
    name='(Ī, Ṝ)',
    marker=dict(color='red', size=10)
))
fig.add_trace(go.Scatter(
    x=[I[-1]], y=[R[-1]],
    mode='markers',
    name='(Iₑ, Rₑ)',
    marker=dict(color='black', size=8)
))
# Layout del grafico
fig.update_layout(
    title='Phase plane (I,R)',
    xaxis_title='Infected (I)',
    yaxis_title='Recovered (R)',
    template='simple_white',
    showlegend=True,
    width=800,
    height=600
)
# Mostra il grafico
fig.show()

## Now Adding Memory

In [ ]:
def beta_func(beta_0, c):
    def beta_function(x):
        return beta_0 / (1 + c * x)

    return beta_function

def SIRS_memory(t, X, beta, gamma, mu, theta, a):
    I,R, M = X

    dI = I * (beta(M) * (1-R-I) - (mu + gamma))
    dR = gamma*I - (mu + theta)*R
    dM = a * (I - M)

    return [dI, dR,  dM]

In [ ]:
mu = m
a = 0.042  # (Characteristic memory length)^-1 (24 months)
beta_0 = 5 * gamma
c = 1
beta = beta_func(beta_0, c)

# Initial conditions
S0 = 0.999  # Initial suspicious population
I0 = 0.001  # Initial infected population
M0 = 0  # Initial memory rate
X0 = [I0,R0, M0]

# t = 1 month
t_span = (0, 50000)
# Solve the system of ODEs
solution_memory = solve_ivp(SIRS_memory, t_span, X0, args=(beta, gamma, mu, theta, a), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol_memory = solution_memory.sol(t)

I_memory = sol_memory[0]
R_memory = sol_memory[1]

In [ ]:

# Troncamento delle serie temporali
x_full = t
y_full = I
y_ref_full = [I_mu_zero[-1]] * len(y_full)

x_post = t[1000:]
y_post = I[1000:]
y_ref_post = [I_mu_zero[-1]] * len(y_post)

# Creazione della figura con subplot (2 colonne)
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.4, 0.6],  # subplot sinistro più stretto
)

# -------------------
fig.add_trace(go.Scatter(
    x=x_full[:1000], y=y_full[:1000],
    mode='lines',
    name='I(t)',
    line=dict(color='rgb(0, 0, 200)')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x_full[:1000], y=y_ref_full[:1000],
    mode='lines',
    name='Ī',
    line=dict(dash='dash', color='red')
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=t[:1000], y=I_memory[:1000],
    mode='lines',
    name='I_mem',
    line=dict(color='green')
), row=1, col=1)

# -------------------
fig.add_trace(go.Scatter(
    x=x_post, y=y_post,
    mode='lines',
    line=dict(color='rgb(0, 0, 200)'),
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_post, y=y_ref_post,
    mode='lines',
    line=dict(dash='dash', color='red'),
    showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=t[1000:], y=I_memory[1000:],
    mode='lines',
    line=dict(color='green'),
    showlegend=False
), row=1, col=2)



# -------------------
# Layout
fig.update_layout(
    width=1000,
    height=600,
    title_text=f'I(t)',
    title_x=0.5,
    template='simple_white',
    showlegend=True
)

# Annotazioni per sostituire i titoli dei subplot (più in basso)
fig.add_annotation(
    text="Up to t=1000",
    xref='paper', yref='paper',
    x=0.2, y=1.01,  # y abbassata
    showarrow=False,
    font=dict(size=14)
)

fig.add_annotation(
    text="From t=1000 onwards",
    xref='paper', yref='paper',
    x=0.8, y=1.01,
    showarrow=False,
    font=dict(size=14)
)

# Annotazioni
fig.add_annotation(
    text=f'Ī: {round(I_mu_zero[-1], 8)}   -    Iₑ: {round(I[-1], 8)}    -    Iₑ / Ī: {round((beta_0 - mu - gamma) / (beta_0 - gamma) * (1 + mu * gamma / (theta ** 2 + theta * (mu + gamma))), 5)}',
    xref='paper', yref='paper',
    x=0.5, y=1.11,
    showarrow=False,
    font=dict(size=13)
)

fig.add_annotation(
    text=f'μ: {round(m, 8)}    -     θ: {round(theta, 8)}     -     γ: {round(gamma, 3)}',
    xref='paper', yref='paper',
    x=0.5, y=1.07,
    showarrow=False,
    font=dict(size=13)
)

# Etichette degli assi
fig.update_xaxes(title_text="Time (days)", row=1, col=1)
fig.update_yaxes(title_text="Infected (I)", row=1, col=1)
fig.update_xaxes(title_text="Time (days)", row=1, col=2)
fig.update_yaxes(title_text="Infected (I)", row=1, col=2)

# Mostra il grafico
fig.show()

In [ ]:
fig = go.Figure()

# Traccia della curva I(t)
fig.add_trace(go.Scatter(
    x=I[1500:], y=R[1500:],
    mode='lines',
    name=' (I, R)',
))
fig.add_trace(go.Scatter(
    x=I_mu_zero[100:], y=R_mu_zero[100:],
    mode='lines',
    name=' (I,R) with μ=0 ',
))
fig.add_trace(go.Scatter(
    x=I_memory[1500:], y=R_memory[1500:],
    mode='lines',
    name='(I, R) with memory ',
))
# Traccia della linea di riferimento costante
fig.add_trace(go.Scatter(
    x=[I_mu_zero[-1]], y=[R_mu_zero[-1]],
    mode='markers',
    name='(Ī, Ṝ)',
    marker=dict(color='red', size=10)
))
fig.add_trace(go.Scatter(
    x=[I[-1]], y=[R[-1]],
    mode='markers',
    name='(Iₑ, Rₑ)',
    marker=dict(color='black', size=8)
))
fig.add_trace(go.Scatter(
    x=[I_memory[-1]], y=[R_memory[-1]],
    mode='markers',
    name='(I_mem, R_mem)',
    marker=dict(color='blue', size=8)
))
# Layout del grafico
fig.update_layout(
    title='Phase plane (I,R)',
    xaxis_title='Infected (I)',
    yaxis_title='Recovered (R)',
    template='simple_white',
    showlegend=True,
    width=800,
    height=600
)
# Mostra il grafico
fig.show()